In [ ]:
import sys
import pandas as pd
from datetime import datetime
sys.path.append('/usr/lib/python3/dist-packages/odoo')
sys.path.append('/mnt/extra-addons')
import pytz
import odoo
import odoo.modules.registry
from odoo import api, tools

local_tz = pytz.timezone('America/Mexico_City')

config = tools.config
config['db_name'] = 'ZTYRES'
config['addons_path'] = '/mnt/extra-addons,/usr/lib/python3/dist-packages/odoo/addons'
# ¡OJO! No uses config['-c']; eso no sirve. Si no vas a parsear, quítalo.

db_name = config['db_name']
registry = odoo.modules.registry.Registry(db_name)
cr = registry.cursor()
env = api.Environment(cr, odoo.SUPERUSER_ID, {})


In [ ]:
from pathlib import Path
from lxml import etree
import re
from typing import List

# Regex razonables para pedimento (15 o 18 dígitos, con o sin espacios)
PED_REGEX_15 = re.compile(r'^\d{2}\s?\d{2}\s?\d{4}\s?\d{7}$')
PED_REGEX_18 = re.compile(r'^\d{2}\s?\d{2}\s?\d{4}\s?\d{7}\s?\d{3}$')
NS = {"cfdi": "http://www.sat.gob.mx/cfd/4", "tfd": "http://www.sat.gob.mx/TimbreFiscalDigital"}

def _get_attr(elem, *names, default=""):
    """Obtiene atributo sin importar mayúsc/minúsc en el nombre."""
    if elem is None:
        return default
    low = {k.lower(): v for k, v in elem.attrib.items()}
    for n in names:
        v = low.get(n.lower())
        if v is not None:
            return v
    return default

def _normalize_pedimento(s: str) -> str:
    """Normaliza a 'AA BB CCCC DDDDDDD [CCC]' cuando hay 15/18 dígitos."""
    digits = re.sub(r'\D', '', s or '')
    if len(digits) >= 15:
        a, b, c, d = digits[:2], digits[2:4], digits[4:8], digits[8:15]
        if len(digits) >= 18:
            e = digits[15:18]
            return f"{a} {b} {c} {d} {e}"
        return f"{a} {b} {c} {d}"
    return s or ''

def _unique_keep_order(seq: List[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def _is_invoice(root) -> bool:
    """Solo facturas (TipoDeComprobante='I')."""
    tdc = _get_attr(root, "TipoDeComprobante", "tipodecomprobante").upper()
    return tdc == "I"

def _safe_date(root) -> str:
    """YYYY-MM-DD si trae hora."""
    f = _get_attr(root, "Fecha", "fecha")
    return f[:10] if f and len(f) >= 10 else f

def _collect_document_level_peds(root) -> List[str]:
    """Busca pedimentos en cualquier InformacionAduanera del documento (fallback)."""
    peds = root.xpath('.//*[local-name()="InformacionAduanera"]/@NumeroPedimento')
    peds = [_normalize_pedimento(p) for p in peds if p]
    peds = [p for p in peds if PED_REGEX_15.match(p) or PED_REGEX_18.match(p)]
    return _unique_keep_order(peds)

def parse_cfdi_folder_concept_pedimento_only(folder: str) -> pd.DataFrame:
    base = Path(folder)
    if not base.exists():
        raise FileNotFoundError(f"No existe la carpeta: {base}")
    
    rows = []
    for xml_path in base.rglob("*.xml"):
        try:
            root = etree.parse(str(xml_path), etree.XMLParser(recover=True, huge_tree=True)).getroot()
        except Exception:
            continue  # XML ilegible

        # Solo Comprobante CFDI 4.0 y facturas (I)
        if root.tag.endswith("Comprobante") and _is_invoice(root):
            fecha = _safe_date(root)
            emisor = root.find("cfdi:Emisor", NS)
            emisor_rfc = _get_attr(emisor, "Rfc")
            emisor_nombre = _get_attr(emisor, "Nombre")

            # NUEVO: serie, folio y uuid
            serie = _get_attr(root, "Serie", "serie")
            folio = _get_attr(root, "Folio", "folio")
            tfd = root.find(".//tfd:TimbreFiscalDigital", NS)
            uuid = _get_attr(tfd, "UUID", "uuid")

            # Recorre conceptos
            for c in root.findall(".//cfdi:Conceptos/cfdi:Concepto", NS):
                # Pedimentos SOLO del concepto (no fallback a nivel documento)
                peds = [ia.get("NumeroPedimento", "").strip()
                        for ia in c.findall("./cfdi:InformacionAduanera", NS)]
                pedimento = " | ".join([p for p in peds if p])  # puede quedar ""

                rows.append({
                    "archivo": xml_path.name,
                    "serie": serie,
                    "folio": folio,
                    "uuid": uuid,
                    "fecha": fecha,
                    "emisor_rfc": emisor_rfc,
                    "emisor_nombre": emisor_nombre,
                    "concepto_clave_prod_serv": c.get("ClaveProdServ", ""),
                    "concepto_no_identificacion": c.get("NoIdentificacion", ""),
                    "concepto_cantidad": c.get("Cantidad", ""),
                    "concepto_clave_unidad": c.get("ClaveUnidad", ""),
                    "concepto_unidad": c.get("Unidad", ""),
                    "concepto_descripcion": c.get("Descripcion", ""),
                    "concepto_valor_unitario": c.get("ValorUnitario", ""),
                    "concepto_importe": c.get("Importe", ""),
                    "pedimento": pedimento,  # vacío si el concepto no trae InformacionAduanera
                })
    
    df = pd.DataFrame(rows)
    df['default_code'] = (
        df['concepto_no_identificacion']
        .astype(str)          # asegura que todo es texto
        .str.lstrip('0')      # elimina ceros a la izquierda
    )
    if not df.empty:
        df = df.reindex(columns=[
            "archivo","serie","folio","uuid","fecha","emisor_rfc","emisor_nombre",
            "default_code","concepto_clave_prod_serv","concepto_no_identificacion","concepto_cantidad",
            "concepto_clave_unidad","concepto_unidad","concepto_descripcion",
            "concepto_valor_unitario","concepto_importe","pedimento"
        ])
    
    df_llantas = df[df['concepto_clave_prod_serv'].astype(str).str.startswith('2517')]
    return df_llantas

RUTA = "CFDIS"  # <-- CAMBIA ESTA RUTA
cfdi_compras = parse_cfdi_folder_concept_pedimento_only(RUTA)

In [ ]:
import pandas as pd
import numpy as np

def _to_numeric_clean(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype(str).str.replace(r'[,\s]', '', regex=True), 
        errors='coerce'
    )

def resumen_por_concepto_fecha_pedimento(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    
    # Tipos
    d['fecha'] = pd.to_datetime(d['fecha'], errors='coerce').dt.date
    for col in ['concepto_cantidad', 'concepto_importe', 'concepto_valor_unitario']:
        if col in d.columns:
            d[col] = _to_numeric_clean(d[col])

    # ¡NO tocar pedimento! (espacios cuentan)
    d['pedimento'] = d['pedimento'].astype('string')

    # Agregamos default_code como clave
    claves = ['default_code','concepto_clave_prod_serv','concepto_no_identificacion', 'concepto_descripcion', 'fecha', 'pedimento']

    agg = (
        d.groupby(claves, dropna=False)
         .agg(
             cantidad_total=('concepto_cantidad', 'sum'),
             importe_total=('concepto_importe', 'sum'),
             xml_distintos=('archivo', 'nunique'),
             uuid_distintos=('uuid', 'nunique'),
             precio_unit_mediana=('concepto_valor_unitario', 'median')
         )
         .reset_index()
    )

    # Promedio ponderado
    agg['precio_unit_prom_pond'] = np.where(
        agg['cantidad_total'].abs() > 0,
        agg['importe_total'] / agg['cantidad_total'],
        np.nan
    )

    # Orden y formato
    agg = agg.sort_values(
        ['default_code','concepto_clave_prod_serv','fecha', 'concepto_no_identificacion', 'pedimento']
    ).reset_index(drop=True)

    for c in ['cantidad_total', 'importe_total', 'precio_unit_mediana', 'precio_unit_prom_pond']:
        if c in agg.columns:
            agg[c] = agg[c].round(2)

    return agg

# Ejecución
resumen = resumen_por_concepto_fecha_pedimento(cfdi_compras)
resumen.to_excel('/mnt/extra-addons/resumen.xlsx')



In [ ]:
# Generas df_products con la data de Odoo
products = env['product.product'].search([])
products_data = []
for product_p in products:
    vals = {
        'product_id': product_p.id,
        'product_default_code': product_p.default_code,
        'product_name': product_p.name,
    }
    products_data.append(vals)

df_products = pd.DataFrame(products_data)

# Columna sin ceros a la izquierda
df_products['default_code_n'] = (
    df_products['product_default_code']
    .astype(str)
    .str.lstrip('0')
)

# --- Hacemos el merge ---
resumen = resumen.merge(
    df_products[['product_id','default_code_n','product_name']], 
    left_on='concepto_no_identificacion', 
    right_on='default_code_n',
    how='left'
)

resumen = resumen.sort_values(
    ['product_id','fecha','concepto_no_identificacion','pedimento']
).reset_index(drop=True)


In [ ]:
resumen['pedimento'] = np.where(
    resumen['pedimento'].isna() | (resumen['pedimento'].str.strip() == ''), 
    'ON', 
    resumen['pedimento']
)
resumen['pedimento'] = resumen['pedimento'].str.replace('|', ',', regex=False)

# Si quieres además asegurarte que no tenga espacios extra
resumen['pedimento'] = resumen['pedimento'].str.replace(' ,', ',', regex=False).str.strip()
df_resumen = resumen

In [ ]:
# ====== INFORME EXCEL DE CAMBIOS (AUTO-CONTENIDO) ======
import io, base64, datetime, re
from collections import defaultdict

try:
    import pandas as pd
except Exception as e:
    raise Exception("Necesitas pandas instalado (pip install pandas).") from e

# -------- Parámetros configurables --------
FIELD_NAME = globals().get('FIELD_NAME', 'pedimento')    # campo en stock.lot
ONLY_POSITIVE_QTY = globals().get('ONLY_POSITIVE_QTY', True)
DRY_RUN = globals().get('DRY_RUN', False)                 # True = solo simula, False = escribe
EXCEL_PATH = globals().get('EXCEL_PATH', "/mnt/extra-addons/informe.xlsx")

# -------- Validación de df_resumen --------
assert 'df_resumen' in globals(), "df_resumen no está definido en el entorno."
for col in ['product_id', 'pedimento', 'cantidad_total']:
    assert col in df_resumen.columns, f"df_resumen requiere columna '{col}'."

# -------- Utilidades y patrones --------
CUSTOM_NUMBERS_PATTERN = re.compile(r"[0-9]{2}\s{2}[0-9]{2}\s{2}[0-9]{4}\s{2}[0-9]{7}")

def _normalize_one(p):
    """Normaliza un pedimento individual al formato AA␠␠BB␠␠BBBB␠␠DDDDDDD; conserva 'ON'."""
    if p is None:
        return ""
    p = str(p).strip()
    if not p:
        return ""
    if p == "ON":
        return "ON"
    digits = re.sub(r"\D", "", p)
    if len(digits) in (15, 18):
        a, b, c, d = digits[:2], digits[2:4], digits[4:8], digits[8:15]
        return f"{a}  {b}  {c}  {d}"
    # Ya viene en 4 bloques numéricos
    p2 = re.sub(r"\s+", " ", p)
    parts = p2.split(" ")
    if len(parts) == 4 and all(part.isdigit() for part in parts):
        return f"{parts[0]}  {parts[1]}  {parts[2]}  {parts[3]}"
    # Si no cumple, regresa limpio
    return p.strip()

def normalize_pedimento(p):
    """
    Soporta múltiples pedimentos separados por ',' o '|'.
    - Normaliza cada uno a AA␠␠BB␠␠BBBB␠␠DDDDDDD
    - Deja 'ON' si no hay números y solo hay ON
    - El resultado final usa coma y espacio como separador
    """
    if p is None:
        return ""
    s = str(p).strip()
    if not s:
        return ""
    parts = re.split(r"[|,]", s)
    normed = []
    saw_digits = False
    for piece in parts:
        piece = piece.strip()
        if not piece:
            continue
        if re.search(r"\d", piece):
            saw_digits = True
        normed_piece = _normalize_one(piece)
        normed.append(normed_piece)
    if not saw_digits:
        # Si no hay números en ninguna parte y apareció ON, deja ON
        if any(x == "ON" for x in normed):
            return "ON"
    # Quita duplicados preservando orden
    seen = set()
    out = []
    for x in normed:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return ", ".join(out)

def pedimento_year(p):
    """Obtiene el año (YYYY) del primer pedimento de una cadena múltiple."""
    if not p:
        return None
    first = str(p).split(",")[0].strip()
    if first == "ON":
        return None
    m = CUSTOM_NUMBERS_PATTERN.search(first)
    if not m:
        digits = re.sub(r"\D", "", first or "")
        if len(digits) < 2:
            return None
        yy = int(digits[:2])
        return 2000 + yy
    yy = int(m.group()[0:2])
    return 2000 + yy

def parse_dot_year(lot):
    """Intenta interpretar lot.name como año (YYYY)."""
    try:
        y = int((lot.name or "").strip())
        return y if 1900 <= y <= 2099 else None
    except Exception:
        return None

# -------- Normaliza df_resumen antes de armar buckets --------
df_resumen = df_resumen.copy()
df_resumen['pedimento'] = (
    df_resumen['pedimento']
    .astype(str)
    .str.replace(r'\|', ',', regex=True)   # convertir '|' a coma
    .apply(normalize_pedimento)
)
df_resumen['cantidad_total'] = df_resumen['cantidad_total'].fillna(0).astype(float)
if 'pedimento_status' not in df_resumen.columns:
    df_resumen['pedimento_status'] = df_resumen['pedimento'].apply(lambda x: 'ON' if str(x).strip() == 'ON' else 'NUM')

# -------- Reconstrucción de buckets --------
buckets = defaultdict(list)
for (pid, ped, status), sub in df_resumen.groupby(['product_id', 'pedimento', 'pedimento_status']):
    remaining = float(sub['cantidad_total'].sum())
    if remaining > 0:
        buckets[int(pid)].append({
            'pedimento': ped,                 # puede contener múltiples separados por coma
            'year': pedimento_year(ped),      # año del primer pedimento
            'status': status,                 # 'ON' o 'NUM'
            'remaining': remaining
        })

# Orden: NUM por año asc (None al final), luego ON
for pid in buckets:
    buckets[pid].sort(key=lambda r: (1 if r['status'] == 'ON' else 0,
                                     r['year'] if r['year'] is not None else 9999))

# -------- Selector de pedimento --------
def pick_pedimento_for_lot(pid, lot_year):
    """
    Lógica:
      1) Si hay NUM con year == lot_year -> ese
      2) Si hay NUM con year > lot_year -> menor de ellos
      3) Si hay cualquier NUM           -> menor year disponible
      4) Si hay ON                      -> ON
    """
    if pid not in buckets:
        return None
    items = buckets[pid]
    avail = [r for r in items if r['remaining'] > 0]
    if not avail:
        return None
    if lot_year is not None:
        eq = [r for r in avail if r['status'] != 'ON' and r['year'] == lot_year]
        if eq:
            eq.sort(key=lambda r: r['year'])
            return eq[0]
        gt = [r for r in avail if r['status'] != 'ON' and (r['year'] is not None and r['year'] > lot_year)]
        if gt:
            gt.sort(key=lambda r: r['year'])
            return gt[0]
    any_num = [r for r in avail if r['status'] != 'ON']
    if any_num:
        any_num.sort(key=lambda r: (r['year'] if r['year'] is not None else 9999))
        return any_num[0]
    any_on = [r for r in avail if r['status'] == 'ON']
    if any_on:
        return any_on[0]
    return None

# -------- Arma 'updates' desde quants (solo ubicaciones internal; qty>0 si aplica) --------
updates = []
Quant = env['stock.quant']
domain = [('lot_id', '!=', False), ('location_id.usage', '=', 'internal')]
if ONLY_POSITIVE_QTY:
    domain.append(('quantity', '>', 0))
quants = Quant.search(domain, order="product_id, lot_id")

for q in quants:
    pid = q.product_id.id
    lot = q.lot_id
    if not lot:
        continue
    # No tocar si ya trae pedimento con texto no vacío
    current_val = getattr(lot, FIELD_NAME, False)
    if str(current_val or "").strip():
        continue
    lot_y = parse_dot_year(lot)
    pick = pick_pedimento_for_lot(pid, lot_y)
    if not pick:
        continue
    qty = float(q.quantity or 0.0)
    if qty <= 0:
        continue
    ped_to_set = pick['pedimento']
    # Consumo simulado del bucket
    pick['remaining'] = max(0.0, pick['remaining'] - qty)
    updates.append((lot.id, ped_to_set, pid, lot.name, qty))

# -------- Snapshot "antes" para remanentes por (product_id, pedimento) --------
snapshot = {}
for pid, lst in buckets.items():
    for r in lst:
        snapshot[(pid, r['pedimento'])] = float(r['remaining'])

# -------- Función para explicar la regla aplicada --------
def _infer_regla(pick, lot_year):
    if not pick:
        return 'sin_coincidencia'
    if pick.get('status') == 'ON':
        return 'ON'
    py = pick.get('year')
    if lot_year is not None and py == lot_year:
        return 'igual_a_anyo_DOT'
    if lot_year is not None and (py is not None) and py > lot_year:
        return 'mayor_a_DOT'
    return 'cualquier_NUM'

# -------- Escritura real en Odoo (si DRY_RUN = False) --------
Lot = env['stock.lot']
Prod = env['product.product']

if not DRY_RUN and updates:
    for (lot_id, ped_after, pid, lot_name, qty) in updates:
        # Normaliza (y soporta múltiples) antes de guardar
        ped_clean = normalize_pedimento(ped_after)
        Lot.browse(lot_id).sudo().write({FIELD_NAME: ped_clean})

# -------- Construcción del reporte --------
rows_report = []

for (lot_id, ped_after, pid, lot_name, qty) in updates:
    lot = Lot.browse(lot_id).sudo()
    prod = Prod.browse(pid).sudo()
    ped_before = getattr(lot, FIELD_NAME, False) or ""

    # localizar info del pedimento en buckets (ya con remaining post-simulación)
    info = None
    for r in buckets.get(pid, []):
        if r['pedimento'] == ped_after:
            info = r
            break

    status_after = (info['status'] if info else ('ON' if ped_after == 'ON' else 'NUM'))
    ped_y = (info['year'] if info else pedimento_year(ped_after))

    # Año DOT desde el nombre del lote
    try:
        lot_y = int((lot_name or '').strip())
        if not (1900 <= lot_y <= 2099):
            lot_y = None
    except Exception:
        lot_y = None

    regla = _infer_regla(info, lot_y)

    rem_before = float(snapshot.get((pid, ped_after), 0.0))
    rem_after = max(0.0, rem_before - float(qty))

    rows_report.append({
        'lot_id': lot_id,
        'lot_name(DOT)': lot_name,
        'product_id': pid,
        'default_code': prod.default_code or '',
        'product_name': prod.display_name or '',
        'qty_quant': float(qty),
        'pedimento_before': ped_before,
        'pedimento_after': normalize_pedimento(ped_after),
        'status_after': status_after,
        'lot_year': lot_y,
        'pedimento_year': ped_y,
        'regla_usada': regla,
        'remanente_before': rem_before,
        'remanente_after': rem_after,
    })

df_detalle = pd.DataFrame(rows_report)

if not df_detalle.empty:
    df_resumen_excel = (
        df_detalle
        .groupby(['product_id', 'default_code', 'product_name', 'pedimento_after', 'status_after'], as_index=False)
        .agg(total_qty=('qty_quant', 'sum'),
             lotes_afectados=('lot_id','nunique'))
        .sort_values(['product_id','default_code','pedimento_after'])
    )
else:
    df_resumen_excel = pd.DataFrame(columns=[
        'product_id','default_code','product_name','pedimento_after','status_after','total_qty','lotes_afectados'
    ])

# -------- Guardar Excel --------
with pd.ExcelWriter(EXCEL_PATH, engine='xlsxwriter') as writer:
    df_detalle.to_excel(writer, sheet_name='Detalle', index=False)
    df_resumen_excel.to_excel(writer, sheet_name='Resumen', index=False)

print(f"[OK] Informe generado en: {EXCEL_PATH}")
print(f"Filas detalle: {len(df_detalle)} | Filas resumen: {len(df_resumen_excel)}")
print(f"DRY_RUN = {DRY_RUN} (False = se guardó en Odoo)")


In [ ]:
resumen.to_excel('/mnt/extra-addons/resumen.xlsx')

In [ ]:
env.cr.commit()